In [0]:
from pyspark.sql.functions import col, sum, count, avg, round

df_orders = spark.read.table("silver_orders")
df_customers = spark.read.table("silver_customers")

df_gold_customer_metrics = (
    df_orders.join(df_customers, "customer_id", "inner")
    .groupBy("customer_id", "country")
    .agg(count("order_id").alias("total_orders"), round(sum("total_amount_usd"), 2).alias("lifetime_spend"), round(avg("total_amount_usd"), 2).alias("avg_order_value"))
)
df_gold_customer_metrics.write.format("delta").mode("overwrite").saveAsTable("gold_customer_metrics")

df_gold_sales_performance = (
    df_orders.groupBy("order_date")
    .agg(count("order_id").alias("daily_order_count"), round(sum("total_amount_usd"), 2).alias("daily_revenue"))
    .orderBy("order_date")
)
df_gold_sales_performance.write.format("delta").mode("overwrite").saveAsTable("gold_sales_performance")